In [9]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [10]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a fitted CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)
    
    for node in bn.nodes:
        cpd = bn.cpds[node]
        num_node_states = cpd.shape[0]
        num_parent_configs = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)
        parents = list(bn.structure.predecessors(node))
        
        if len(parents) == 0:
            probs = data[node].map(cpd.iloc[:, 0]).values
        else:
            cpd_series = cpd.unstack()
            parent_order = sorted(parents)
            lookup_cols = parent_order + [node]
            lookup_df = data[lookup_cols]
            lookup_keys = [tuple(x) for x in lookup_df.to_numpy()]
            probs = cpd_series.loc[lookup_keys].values
            
        log_likelihood += np.sum(np.log(probs + 1e-9))
    
    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [11]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [12]:
def discretize_data(df):
    discretised_df = pd.DataFrame(index=df.index)
    TARGET_BINS = 3

    for col in df.columns:
        if df[col].isna().all():
            discretised_df[col] = "Missing"
            continue
        
        no_unique = df[col].nunique()
    
        if no_unique <= 1:
            discretised_df[col] = "Constant"
            continue
    
        if no_unique <= TARGET_BINS:
            discretised_df[col] = df[col].rank(method='dense') - 1
        else:
            try:
                discretised_df[col] = pd.qcut(df[col], q=TARGET_BINS, labels=False, duplicates='drop')
            except (ValueError, TypeError):
                try:
                    discretised_df[col] = pd.cut(df[col], bins=TARGET_BINS, labels=False, include_lowest=True)
                except (ValueError, TypeError):
                    discretised_df[col] = pd.qcut(df[col].rank(method='first'), q=TARGET_BINS, labels=False, duplicates='drop')

        if pd.api.types.is_numeric_dtype(discretised_df[col]):
            valid_values = sorted(discretised_df[col].dropna().unique())
            num_states = len(valid_values)
            
            if num_states == 2:
                label_map = {valid_values[0]: "Low", valid_values[1]: "High"}
            elif num_states == 3:
                label_map = {valid_values[0]: "Low", valid_values[1]: "Medium", valid_values[2]: "High"}
            elif num_states == 4:
                label_map = {valid_values[0]: "VeryLow", valid_values[1]: "Low", valid_values[2]: "High", valid_values[3]: "VeryHigh"}
            else:
                label_map = {val: f"State{int(val)}" for val in valid_values}
            
            discretised_df[col] = discretised_df[col].map(label_map)
            
            discretised_df[col] = discretised_df[col].fillna("Unknown")

    print("Data types after discretization:")
    print(discretised_df.dtypes.value_counts())

    print("\nUnique states per column:")
    print(discretised_df.nunique())

    print("\nDiscretised data preview:")
    print(discretised_df.head())

    if discretised_df.isna().any().any():
        print("\n⚠️ Warning: Some NaN values remain!")
        print(discretised_df.isna().sum()[discretised_df.isna().sum() > 0])
    
    return discretised_df

In [13]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_bn_no_context.csv")

df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

df_pl

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8790.0,29.28,67.985714,3.630000,2.570000,21.39,6.903,-3.803166,18.166685,-1.872662,5.83,-0.3,17.3,0.000009,62.749213,36.747184
1,9180.0,28.82,67.985714,3.690000,1.910000,17.10,7.686,-5.237136,18.626276,-3.397579,2.72,0.5,16.9,0.000009,65.250459,57.512070
2,9410.0,29.96,62.600000,3.740000,2.060000,20.47,8.676,-0.960496,18.707747,-6.157262,3.18,0.8,17.1,0.000008,75.989515,135.142137
3,9740.0,29.70,62.200000,3.650000,2.230000,23.58,9.281,-2.441439,19.149239,-6.142614,3.94,1.0,17.6,0.000009,84.200841,180.098848
4,10250.0,30.39,62.500000,3.650000,2.330000,25.53,10.337,-2.287307,18.077861,-4.112383,3.57,0.0,17.7,0.000009,101.918024,247.386988
5,10400.0,30.23,62.900000,3.810000,2.620000,26.31,10.955,-0.784100,18.008657,-3.437896,1.53,0.0,17.1,0.000008,115.869679,265.323092
6,10480.0,28.68,63.200000,3.910000,3.050000,27.16,11.452,1.117069,18.330801,-4.023830,0.26,-1.2,17.3,0.000008,111.382755,275.386821
7,10900.0,29.94,64.900000,4.020000,3.550000,23.73,11.605,0.407324,18.238027,-3.367862,3.85,-0.3,17.0,0.000008,103.683241,307.471216
8,11390.0,30.99,66.300000,4.070000,3.810000,20.68,11.881,2.067846,17.905031,-2.532527,3.30,-1.0,17.6,0.000008,123.085022,169.995130
9,11740.0,30.67,68.200000,4.300000,4.000000,17.59,11.396,2.946955,17.821550,-2.650770,3.82,0.2,17.3,0.000008,112.220174,76.266802


In [14]:
pl = pd.DataFrame()
pl = df_pl.copy()

In [15]:
pl = pl.dropna()

In [18]:
discretised_pl = discretize_data(pl)

Data types after discretization:
object    16
Name: count, dtype: int64

Unique states per column:
GDPC    3
VABI    3
EMP     3
QOA     3
QOR     3
YUA     3
REST    3
EBSG    3
GGFC    3
LBGD    3
FDI     3
PC      3
ARP     3
CO2C    3
PAM     3
EUPC    3
dtype: int64

Discretised data preview:
  GDPC    VABI     EMP  QOA  QOR     YUA REST EBSG    GGFC    LBGD     FDI  \
0  Low  Medium  Medium  Low  Low  Medium  Low  Low  Medium    High    High   
1  Low     Low  Medium  Low  Low  Medium  Low  Low    High  Medium     Low   
2  Low    High     Low  Low  Low  Medium  Low  Low    High     Low     Low   
3  Low  Medium     Low  Low  Low    High  Low  Low    High     Low    High   
4  Low    High     Low  Low  Low    High  Low  Low  Medium     Low  Medium   

       PC     ARP    CO2C     PAM    EUPC  
0  Medium  Medium    High     Low     Low  
1    High  Medium  Medium     Low     Low  
2    High  Medium     Low     Low     Low  
3    High    High    High     Low  Medium  
4  Medium   

In [19]:
sm = StructureModel()

In [20]:
sm.add_edges_from([
    ('PAM', 'EUPC')
])

In [21]:
sm.edges

OutEdgeView([('PAM', 'EUPC')])

In [22]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [23]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_pl)
bn_baseline.fit_cpds(discretised_pl, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_pl)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (PAM -> EUPC)
# ==========================================
print("\n--- Testing PAM -> EUPC ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['PAM', 'EUPC'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

BASELINE BIC: 89.81

--- Testing PAM -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['PAM', 'EUPC']
  - Removing direct edge: PAM -> EUPC
LV Model BIC: 99.79 (LogLikelihood: -37.42)


In [25]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_bn_no_context.csv")

df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

df_pl

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8790.0,29.28,67.985714,3.630000,2.570000,21.39,6.903,-3.803166,18.166685,-1.872662,5.83,-0.3,17.3,0.000009,62.749213,36.747184
1,9180.0,28.82,67.985714,3.690000,1.910000,17.10,7.686,-5.237136,18.626276,-3.397579,2.72,0.5,16.9,0.000009,65.250459,57.512070
2,9410.0,29.96,62.600000,3.740000,2.060000,20.47,8.676,-0.960496,18.707747,-6.157262,3.18,0.8,17.1,0.000008,75.989515,135.142137
3,9740.0,29.70,62.200000,3.650000,2.230000,23.58,9.281,-2.441439,19.149239,-6.142614,3.94,1.0,17.6,0.000009,84.200841,180.098848
4,10250.0,30.39,62.500000,3.650000,2.330000,25.53,10.337,-2.287307,18.077861,-4.112383,3.57,0.0,17.7,0.000009,101.918024,247.386988
5,10400.0,30.23,62.900000,3.810000,2.620000,26.31,10.955,-0.784100,18.008657,-3.437896,1.53,0.0,17.1,0.000008,115.869679,265.323092
6,10480.0,28.68,63.200000,3.910000,3.050000,27.16,11.452,1.117069,18.330801,-4.023830,0.26,-1.2,17.3,0.000008,111.382755,275.386821
7,10900.0,29.94,64.900000,4.020000,3.550000,23.73,11.605,0.407324,18.238027,-3.367862,3.85,-0.3,17.0,0.000008,103.683241,307.471216
8,11390.0,30.99,66.300000,4.070000,3.810000,20.68,11.881,2.067846,17.905031,-2.532527,3.30,-1.0,17.6,0.000008,123.085022,169.995130
9,11740.0,30.67,68.200000,4.300000,4.000000,17.59,11.396,2.946955,17.821550,-2.650770,3.82,0.2,17.3,0.000008,112.220174,76.266802


In [26]:
pl = pd.DataFrame()
pl = df_pl.copy()
pl['VABI_1'] = pl['VABI'].shift(1)

In [27]:
pl = pl.dropna()

In [28]:
discretised_pl = discretize_data(pl)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
VABI_1    3
dtype: int64

Discretised data preview:
  GDPC    VABI     EMP  QOA  QOR     YUA REST EBSG    GGFC    LBGD     FDI  \
1  Low     Low  Medium  Low  Low  Medium  Low  Low    High  Medium     Low   
2  Low    High     Low  Low  Low  Medium  Low  Low    High     Low  Medium   
3  Low  Medium     Low  Low  Low    High  Low  Low    High     Low    High   
4  Low    High     Low  Low  Low    High  Low  Low  Medium     Low  Medium   
5  Low    High     Low  Low  Low    High  Low  Low  Medium  Medium     Low   

       PC     ARP    CO2C     PAM    EUPC  VABI_1  
1    High  Medium  Medium     Low     Low  Medium  
2    High  Medium     Low     Low     Low     Low  
3    High    High    High     Lo

In [29]:
sm = StructureModel()

In [30]:
sm.add_edges_from([
    ('EUPC', 'REST'),
    ('EUPC', 'PAM'),
    ('VABI_1', 'EUPC')
    
])

In [31]:
sm.edges

OutEdgeView([('EUPC', 'REST'), ('EUPC', 'PAM'), ('VABI_1', 'EUPC')])

In [32]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [34]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_pl)
bn_baseline.fit_cpds(discretised_pl, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_pl)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> REST)
# ==========================================
print("\n--- Testing EUPC -> REST ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'REST'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

# ==========================================
# TEST #2: (EUPC -> PAM)
# ==========================================
print("\n--- Testing EUPC -> PAM ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['EUPC', 'PAM'], 
    discretised_data=discretised_pl,
    raw_data=pl
)

# ==========================================
# TEST: (VABI_1 -> EUPC)
# ==========================================
print("\n--- Testing VABI_1 -> EUPC ---")

sm_test_3 = sm_baseline.copy()

bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_3,
    children_cols=['VABI_1', 'EUPC'], 
    discretised_data=discretised_pl,
    raw_data=pl
)



BASELINE BIC: 168.53

--- Testing EUPC -> REST ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'REST']
  - Removing direct edge: EUPC -> REST
LV Model BIC: 192.79 (LogLikelihood: -62.55)

--- Testing EUPC -> PAM ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'PAM']
  - Removing direct edge: EUPC -> PAM
LV Model BIC: 190.51 (LogLikelihood: -61.41)

--- Testing VABI_1 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['VABI_1', 'EUPC']
  - Removing direct edge: VABI_1 -> EUPC
LV Model BIC: 180.45 (LogLikelihood: -61.79)
